# Model Quantisation — A Practical, Hands-On Guide

### Run Large Language Models on your own laptop (no expensive GPU needed)

This notebook teaches **model quantisation** step by step and shows you how to **quantise Qwen3.5-4B with
NF4 + double quantisation**, run it locally, and understand **KV cache, prefill/decode and PagedAttention**.

**What you will learn**
1. What quantisation is and the number formats (FP32 / FP16 / INT8 / INT4)
2. *Why* we quantise — smaller, cheaper, runs offline
3. The intuition with a tiny hands-on example
4. Types of quantisation (PTQ vs QAT) and the popular tools
5. **NF4 + double quantisation** — the maths, and a live demo on a tensor
6. **Hands-on:** quantise Qwen3.5, save it locally, and look inside
7. **Inference internals:** KV cache, prefill vs decode, PagedAttention
8. The companion CLI and the **FastAPI** server

> **Audience:** beginners to intermediate. Every concept is explained in plain language with runnable code.

## 1. What is quantisation?

A neural network is just a huge pile of **numbers** (called *weights*). By default each number is stored as a **32-bit floating point** value (FP32) — that is **4 bytes** per number.

**Quantisation = storing those numbers using fewer bits.**

Instead of 32 bits per weight, we use 16, 8, or even 4 bits. We trade a tiny bit of accuracy for a *huge* saving in memory and a nice speed-up.

| Format | Bits | Bytes/weight | Typical use |
|--------|------|--------------|-------------|
| FP32   | 32   | 4.0          | Training / original weights |
| FP16 / BF16 | 16 | 2.0       | GPU inference, half the size |
| INT8   | 8    | 1.0          | 4x smaller, very common |
| INT4   | 4    | 0.5          | 8x smaller, run big models on small HW |

### The memory math
Memory ≈ **number of parameters × bytes per parameter**.

A 7-billion-parameter model:
- FP32: 7B × 4 = **28 GB**  (won't fit on most laptops)
- FP16: 7B × 2 = **14 GB**
- INT8: 7B × 1 = **7 GB**
- INT4: 7B × 0.5 = **3.5 GB**  (fits on a normal laptop!)

That is the whole reason quantisation matters: it makes models that needed a data-centre GPU run on the hardware you already own.

In [ ]:
# A quick calculator: how big is a model at each precision?
def model_memory(num_params_billions, bytes_per_param):
    return num_params_billions * 1e9 * bytes_per_param / (1024**3)  # in GB

for size in [0.5, 7, 13, 70]:
    print(f"\n{size}B parameter model:")
    for name, b in [("FP32", 4), ("FP16", 2), ("INT8", 1), ("INT4", 0.5)]:
        print(f"   {name:5s}: {model_memory(size, b):7.2f} GB")

## 2. Why quantise? (the benefits)

1. **Smaller** – 4x–8x less disk and RAM. Big models fit on small machines.
2. **Faster** – moving fewer bytes from memory to the CPU/GPU is the main bottleneck in LLM inference; less data = faster tokens.
3. **Cheaper** – no need to rent a cloud GPU; run on the laptop you have.
4. **Private & offline** – the model runs **on your machine**, so your data never leaves it.
5. **Greener** – less compute = less energy.

**The cost:** a small drop in accuracy. For INT8 it's usually negligible, which is why this guide uses **INT8**: the safest trade-off between size, speed and quality.

## 3. The intuition — quantise a few numbers by hand

Quantisation maps a range of real numbers onto a small set of integers.

For INT8 we have 256 possible values (−128..127). The recipe (symmetric quantisation):

1. Find the largest absolute value in the tensor → `absmax`.
2. `scale = absmax / 127`.
3. `q = round(x / scale)`  → store these small integers.
4. To use the weights again: `x_approx = q * scale`  ("dequantise").

Run the cell below to see the tiny rounding error this introduces.

In [ ]:
import numpy as np

# Pretend these are some model weights (FP32).
weights = np.array([0.12, -0.85, 1.73, -0.04, 0.66, -1.20], dtype=np.float32)

# --- Quantise to INT8 (symmetric) ---
absmax = np.abs(weights).max()
scale  = absmax / 127.0
q      = np.round(weights / scale).astype(np.int8)      # the stored 8-bit integers

# --- Dequantise (what the model actually uses) ---
deq = q.astype(np.float32) * scale

print("original (fp32):", weights)
print("quantised (int8):", q)
print("dequantised     :", np.round(deq, 4))
print("max error       :", np.abs(weights - deq).max())
print(f"\nStored as int8 -> 1 byte each instead of 4 bytes: {weights.nbytes} -> {q.nbytes} bytes")

## 4. Types & tools of quantisation

### Two big families
- **PTQ – Post-Training Quantisation:** take an already-trained model and shrink it. Fast, cheap, no retraining. **This is what we use here.**
- **QAT – Quantisation-Aware Training:** simulate quantisation *during* training for the best accuracy. Powerful but expensive — not needed for running models locally.

### Popular PTQ tools
| Tool | Bits | Best for |
|------|------|----------|
| **bitsandbytes** (NF4 + double quant) | 4 | Quantise on load, QLoRA fine-tuning; runs on GPU and CPU |
| **GPTQ** / **AWQ** | 4 | High-quality 4-bit on GPU (needs calibration data) |
| **GGUF / llama.cpp** | 2–8 | Optimised CPU inference |

This guide uses **bitsandbytes NF4 + double quantisation**: no calibration data needed, one line of config,
and it is the method behind QLoRA.

## 5. NF4 + double quantisation

**NF4 (4-bit NormalFloat).** Trained weights are roughly *normally distributed* (most near 0, few large).
INT4 spaces its 16 levels evenly; NF4 places its 16 levels at the **quantiles of a normal distribution**,
so more levels sit where most weights are. Weights are quantised in **blocks of 64**, each with one
scale factor (`absmax`).

**Double quantisation** (from the QLoRA paper, 2023). Those scale factors are FP32: 32 bits / 64 weights =
**0.5 extra bits per weight**. Double quantisation quantises the scale factors too (to 8-bit, in blocks of 256):

| | weights | scale factors | total |
|---|---|---|---|
| NF4 | 4 bits | 32/64 = **0.5** bits | 4.5 bits |
| NF4 + double quant | 4 bits | 8/64 + 32/(64·256) = **0.127** bits | 4.127 bits |

In [ ]:
single = 32 / 64
double = 8 / 64 + 32 / (64 * 256)
print(f"Scale overhead: NF4 {single:.3f} bits/param, NF4+DQ {double:.3f} bits/param, saved {single - double:.3f}")
for billions in [0.8, 4, 7, 70]:
    print(f"  {billions:>4}B model -> double quant saves {billions * 1e9 * (single - double) / 8 / 1024**3:.2f} GB")

### See it on a real tensor (`compress_statistics=True` is bitsandbytes' name for double quantisation)

In [ ]:
import torch
import bitsandbytes.functional as F

torch.manual_seed(0)
W = torch.randn(4096, 4096)  # a pretend weight matrix (16.7M params)

q1, s1 = F.quantize_4bit(W, quant_type="nf4", compress_statistics=False)
q2, s2 = F.quantize_4bit(W, quant_type="nf4", compress_statistics=True)


def scale_bytes(s):
    return s.absmax.numel() * s.absmax.element_size() + (s.state2.absmax.numel() * 4 if s.nested else 0)


w_bytes = q1.numel() * q1.element_size()
for name, (q, s) in {"NF4": (q1, s1), "NF4 + double quant": (q2, s2)}.items():
    bits = (w_bytes + scale_bytes(s)) * 8 / W.numel()
    err = (F.dequantize_4bit(q, s) - W).abs().mean().item()
    print(f"{name:<20} scales {s.absmax.dtype}  {bits:.3f} bits/param  mean error {err:.5f}")

The error is practically identical: only the *scale factors* were rounded, not the weights.

> **On a CPU, NF4 is about memory, not speed.** bitsandbytes de-quantises the 4-bit weights back to float for
> each matrix multiply, so it is usually *slower* than an unquantised float32 model, but needs far less RAM.
> We compute in **float32** because most laptop CPUs have no fast bfloat16 math (~5x slower in testing).

## 6. Hands-on: quantise Qwen3.5 with NF4 + double quantisation

`Qwen/Qwen3.5-0.8B` keeps this quick (~1.7 GB download); switch `MODEL` to `"Qwen/Qwen3.5-4B"` for the real
thing (~9 GB download).

> Use the Hugging Face id `Qwen/Qwen3.5-4B`, **not** the Ollama tag `qwen3.5:4b`.

In [ ]:
from model_quantization.config import default_output_dir
from model_quantization.quantize import quantize

MODEL = "Qwen/Qwen3.5-0.8B"                      # or "Qwen/Qwen3.5-4B"
OUT = default_output_dir(MODEL)                  # <MODELS_DIR from .env>/<model>-nf4-dq

if not (OUT / "config.json").exists():
    quantize(MODEL, OUT)
print("NF4-dq model folder:", OUT.resolve())

### Look inside: 4-bit weights with double-quantised scales

In [ ]:
import json
from model_quantization.engine import Engine

engine = Engine(OUT)
model = engine.model
linear = model.get_decoder().layers[0].mlp.down_proj
qs = linear.weight.quant_state
print("layer type     :", type(linear).__name__)
print("4-bit storage  :", linear.weight.dtype, tuple(linear.weight.shape), "(two 4-bit values per byte)")
print("quant type     :", qs.quant_type, "| block size", qs.blocksize)
print("double quant   :", qs.nested, "| scales stored as", qs.absmax.dtype)
print("compute dtype  :", model.dtype)
print("saved config   :", json.loads((OUT / "config.json").read_text())["quantization_config"])
print(f"size on disk   : {sum(f.stat().st_size for f in OUT.rglob('*') if f.is_file()) / 1024**3:.2f} GB")

## 7. Inference internals: KV cache, prefill vs decode, PagedAttention

**Prefill vs decode.** Generating an answer has two phases:
1. **Prefill** — the whole prompt goes through the model in **one** forward pass. Many tokens at once, so it
   is *compute-bound* and fast per token. Its time is the **time to first token (TTFT)**.
2. **Decode** — then one token per step. Every step reads all the weights for a single token, so it is
   *memory-bound* and much slower per token.

**KV cache.** Attention needs the keys/values of every earlier token. Without a cache, each decode step
recomputes them for the whole sequence (O(n²) work). With a cache, prefill stores them once and each decode
step adds just one token's K/V.

**PagedAttention** (vLLM). Instead of one growing tensor per request, the cache is a pre-allocated pool of
fixed-size **blocks** (16 tokens here). Each request has a **block table** (logical block → physical block),
like virtual-memory pages: no fragmentation, at most one partly-filled block wasted, and blocks go back to the
pool when a request ends — which is what lets a server batch many users.

Qwen3.5 is a *hybrid* model: only its full-attention layers (8 of 32 in the 4B model) have a KV cache; the
others are linear attention with a small fixed-size state. Our paged cache is in `src/model_quantization/kv_cache.py`.

In [ ]:
pool = engine.pool
print(f"KV pool: {pool.num_blocks} blocks x {pool.block_size} tokens = {pool.total_mb:.0f} MB "
      f"({pool.bytes_per_token / 1024:.0f} KB per token)")

prompt = "What is model quantisation? Answer in 2 short sentences."
for mode in ["paged", "dynamic", "none"]:
    g = engine.generate(prompt, max_new_tokens=30, kv_cache=mode)
    extra = f" | {g.cache_blocks} blocks = {g.cache_mb:.1f} MB" if mode == "paged" else ""
    print(f"{mode:<8} TTFT {g.ttft:5.2f}s ({g.prefill_tps:5.1f} tok/s prefill) | "
          f"decode {g.decode_tps:5.2f} tok/s{extra}")
print("\nAnswer:", g.text)
print("Blocks in use after the requests:", pool.used_blocks, "(all returned to the pool)")

`paged` and `dynamic` give the same answer at a similar speed (batch size 1); `none` gets slower and slower as
the answer grows, because it re-processes the whole sequence for every new token.

## 8. The companion CLI and FastAPI server

The same steps from a terminal (run from the project root):

```bash
uv run model-quantization quantize                       # Qwen3.5-4B -> models/qwen3.5-4b-nf4-dq
uv run model-quantization run "What is NF4?"             # prints prefill / decode / KV-cache stats
uv run model-quantization run "What is NF4?" --kv-cache none
uv run model-quantization serve                          # http://127.0.0.1:8000/docs
uv run model-quantization status                         # what is running + RAM
uv run model-quantization stop
```

Call the API:
```bash
curl -X POST http://127.0.0.1:8000/generate -H "Content-Type: application/json" \
     -d '{"prompt": "What is double quantisation?", "max_new_tokens": 100, "kv_cache": "paged"}'
```